# 13 — EEGNet LOSO Interpretability

This notebook generates LOSO EEGNet interpretability artifacts for the thesis.

It uses the manifest exported by Notebook 12 and, for each selected held-out subject, loads:

- the trained EEGNet checkpoint
- the train-derived normalization statistics
- the held-out evaluation cache
- saved probabilities and predictions

It then computes Integrated Gradients on selected held-out trials and produces:

- per-subject broadband attribution figures
- per-subject band-limited attribution figures
- per-subject trial-consistency figures
- subject-level summary tables
- subject-averaged LOSO summary figures and tables

The notebook is manifest-driven and should not reconstruct checkpoint/cache paths manually.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import mne

from scipy.signal import butter, sosfiltfilt
from IPython.display import display

from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe, save_experiment_metadata, save_json
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
    valid_topomap_picks,
)


warnings.filterwarnings("ignore")

set_pandas_display_options()

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE      :", DEVICE)

In [ ]:
# EXPERIMENT_NAME = "ds005863_loso_main"
EXPERIMENT_NAME = "ds003061_loso_main"

cfg_loso = get_experiment(EXPERIMENT_NAME)

EXPORT_ROOT = cfg_loso.artifact_dir("eegnet_loso_exports")
MANIFEST_PATH = EXPORT_ROOT / "manifest.csv"
CONFIG_SNAPSHOT_PATH = EXPORT_ROOT / "config_snapshot.json"

OUT_DIR = cfg_loso.artifact_dir("eegnet_loso_interpretability")
FIG_ROOT = OUT_DIR / "figures"
SUMMARY_FIG_ROOT = FIG_ROOT / "summary"
TABLE_ROOT = OUT_DIR / "tables"

FIG_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_FIG_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)

save_experiment_metadata(
    cfg_loso,
    extra={
        "notebook": "13_eegnet_loso_interpretability.ipynb",
        "purpose": "Load LOSO EEGNet export packages and generate per-subject plus subject-averaged interpretability artifacts.",
        "experiment_key": EXPERIMENT_NAME,
        "export_root": str(EXPORT_ROOT),
        "manifest_path": str(MANIFEST_PATH),
        "artifact_dir": str(OUT_DIR),
    },
)

print("Experiment key:", EXPERIMENT_NAME)
print("Export root   :", EXPORT_ROOT)
print("Manifest      :", MANIFEST_PATH)
print("Output dir    :", OUT_DIR)
print("Figure root   :", FIG_ROOT)
print("Table root    :", TABLE_ROOT)

In [ ]:
P300_WINDOW = (0.25, 0.50)
PRESTIM_WINDOW = (-0.2, 0.0)
BASELINE_TIME = 0.0

TARGET_LABEL = 1
PROB_THRESHOLD = 0.50

ATTR_BATCH_SIZE = 8
IG_STEPS = 32

MAX_TRIALS_PER_SUBJECT = 64
MIN_CORRECT_TARGETS = 12

BASELINE_MODE = "prestim_mean"  # options: "zeros", "prestim_mean"

BANDS = {
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "gamma": (30.0, 45.0),
}

# Options:
# - "best_auc": choose best seed per subject from manifest
# - integer: force a seed, for example 42
SEED_SELECTION = "best_auc"

# Example: SUBJECT_INCLUDE = ["001", "006"]
SUBJECT_INCLUDE = None

print("P300 window      :", P300_WINDOW)
print("Baseline mode    :", BASELINE_MODE)
print("IG steps         :", IG_STEPS)
print("Seed selection   :", SEED_SELECTION)
print("Subject include  :", SUBJECT_INCLUDE)

In [ ]:
def normalize_subject_id(x) -> str:
    """
    Convert subject identifiers such as '1', '001', 'sub-001' into '001'.
    """
    s = str(x)
    m = re.search(r"(\d{1,3})", s)

    if not m:
        raise ValueError(f"Could not parse subject id from: {x}")

    return f"{int(m.group(1)):03d}"


def get_window_mask(times: np.ndarray, tmin: float, tmax: float) -> np.ndarray:
    times = np.asarray(times, dtype=float)
    return (times >= float(tmin)) & (times <= float(tmax))


def p300_mask_from_times(times: np.ndarray) -> np.ndarray:
    mask = get_window_mask(times, P300_WINDOW[0], P300_WINDOW[1])

    if mask.sum() == 0:
        raise ValueError(
            f"P300 window {P300_WINDOW} has no samples. "
            f"Time range is {times[0]} to {times[-1]}."
        )

    return mask


def pick_evenly_spaced_indices(indices, max_n: int) -> np.ndarray:
    indices = np.asarray(indices, dtype=int)

    if len(indices) <= int(max_n):
        return indices

    pick = np.linspace(0, len(indices) - 1, int(max_n)).round().astype(int)
    return indices[pick]


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return float("nan")

In [ ]:
def load_eval_data_from_npz(npz_path: Path) -> dict:
    """
    Load one held-out evaluation cache exported by Notebook 12.
    """
    npz_path = Path(npz_path)

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing held-out cache: {npz_path}")

    raw = np.load(npz_path, allow_pickle=True)
    data = {k: raw[k] for k in raw.files}

    required = ["X", "y", "times", "ch_names", "sfreq"]
    missing = [k for k in required if k not in data]

    if missing:
        raise KeyError(f"Missing keys in {npz_path}: {missing}")

    X = np.asarray(data["X"], dtype=np.float32)
    y = np.asarray(data["y"]).astype(int)
    times = np.asarray(data["times"], dtype=float)
    ch_names = [str(c) for c in np.asarray(data["ch_names"]).tolist()]
    sfreq = float(np.asarray(data["sfreq"]).item())

    subjects = (
        np.asarray(data["subjects"]).astype(str)
        if "subjects" in data
        else np.asarray(["000"] * len(X), dtype=str)
    )
    subjects = np.asarray([normalize_subject_id(s) for s in subjects], dtype=str)

    out = {
        "X": X,
        "y": y,
        "times": times,
        "ch_names": ch_names,
        "sfreq": sfreq,
        "subjects": subjects,
        "groups": np.asarray(data["groups"]).astype(int) if "groups" in data else None,
        "sample_idx": np.asarray(data["sample_idx"]).astype(int) if "sample_idx" in data else None,
        "prob": np.asarray(data["prob"], dtype=float) if "prob" in data else None,
        "pred": np.asarray(data["pred"]).astype(int) if "pred" in data else None,
        "correct": np.asarray(data["correct"]).astype(int) if "correct" in data else None,
    }

    return out

In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}\n"
        "Run Notebook 12 first."
    )

manifest_df = pd.read_csv(MANIFEST_PATH)

required_manifest_cols = {
    "subject",
    "seed",
    "test_auc",
    "checkpoint_path",
    "heldout_eval_path",
}

missing_cols = required_manifest_cols - set(manifest_df.columns)

if missing_cols:
    raise ValueError(
        "Manifest is missing required columns: "
        + ", ".join(sorted(missing_cols))
    )

manifest_df["subject"] = manifest_df["subject"].astype(str).map(normalize_subject_id)
manifest_df["seed"] = manifest_df["seed"].astype(int)
manifest_df["test_auc"] = manifest_df["test_auc"].astype(float)

if SUBJECT_INCLUDE is not None:
    include_set = {normalize_subject_id(s) for s in SUBJECT_INCLUDE}
    manifest_df = manifest_df[manifest_df["subject"].isin(include_set)].copy()

if len(manifest_df) == 0:
    raise RuntimeError("Manifest is empty after filtering.")

if SEED_SELECTION == "best_auc":
    chosen_df = (
        manifest_df
        .sort_values(["subject", "test_auc"], ascending=[True, False])
        .groupby("subject", as_index=False)
        .first()
        .sort_values("subject")
        .reset_index(drop=True)
    )
elif isinstance(SEED_SELECTION, int):
    chosen_df = (
        manifest_df[manifest_df["seed"] == int(SEED_SELECTION)]
        .sort_values("subject")
        .reset_index(drop=True)
    )

    if len(chosen_df) == 0:
        raise RuntimeError(f"No manifest rows found for seed={SEED_SELECTION}")
else:
    raise ValueError("SEED_SELECTION must be 'best_auc' or an integer seed.")

SUBJECT_CONFIGS = {}

for _, row in chosen_df.iterrows():
    sid = normalize_subject_id(row["subject"])

    SUBJECT_CONFIGS[sid] = {
        "subject": sid,
        "seed": int(row["seed"]),
        "auc": float(row["test_auc"]),
        "checkpoint_path": Path(row["checkpoint_path"]),
        "heldout_eval_path": Path(row["heldout_eval_path"]),
        "history_path": Path(row["history_path"]) if "history_path" in row and pd.notna(row["history_path"]) else None,
        "metrics_path": Path(row["metrics_path"]) if "metrics_path" in row and pd.notna(row["metrics_path"]) else None,
    }

print("Chosen exports:", len(chosen_df))
display(chosen_df[["subject", "seed", "test_auc", "checkpoint_path", "heldout_eval_path"]])

In [ ]:
def strip_module_prefix(state_dict: dict) -> dict:
    out = {}

    for key, value in state_dict.items():
        if key.startswith("module."):
            out[key[len("module."):]] = value
        else:
            out[key] = value

    return out


def extract_state_dict(ckpt: dict) -> dict:
    """
    Accepts several common checkpoint layouts.
    Notebook 12 saves the model under 'state_dict'.
    """
    if not isinstance(ckpt, dict):
        raise ValueError("Checkpoint must be a dictionary.")

    for key in ["state_dict", "model_state_dict"]:
        if key in ckpt and isinstance(ckpt[key], dict):
            return strip_module_prefix(ckpt[key])

    if all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        return strip_module_prefix(ckpt)

    raise ValueError("Could not extract model state_dict from checkpoint.")


def to_numpy_norm(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().astype(np.float32)

    return np.asarray(x, dtype=np.float32)


def load_model_and_norm(checkpoint_path: Path, n_channels: int, n_times: int):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if "norm_mean" not in ckpt or "norm_std" not in ckpt:
        raise KeyError(
            f"Checkpoint does not contain norm_mean/norm_std: {checkpoint_path}"
        )

    model_kwargs = dict(ckpt.get("model_kwargs", {}))

    model = models.get_model(
        "eegnet",
        n_ch=int(n_channels),
        n_times=int(n_times),
        eegnet_kwargs=model_kwargs,
    )

    state_dict = extract_state_dict(ckpt)
    model.load_state_dict(state_dict, strict=True)

    model.to(DEVICE)
    model.eval()

    norm_mean = to_numpy_norm(ckpt["norm_mean"])
    norm_std = to_numpy_norm(ckpt["norm_std"])

    print("Loaded checkpoint:", checkpoint_path.name)
    print("Model kwargs     :", model_kwargs)
    print("norm_mean shape  :", norm_mean.shape)
    print("norm_std shape   :", norm_std.shape)

    return model, ckpt, norm_mean, norm_std


def normalize_trials(X: np.ndarray, norm_mean: np.ndarray, norm_std: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    return ((X - norm_mean) / (norm_std + eps)).astype(np.float32)


def prepare_model_input(x: torch.Tensor) -> torch.Tensor:
    """
    Project EEGNet convention:
        input to dataset/model before unsqueeze: (N, C, T)
        EEGNet input: (N, 1, C, T)
    """
    if x.ndim != 3:
        raise ValueError(f"Expected tensor shape (N, C, T), got {tuple(x.shape)}")

    return x.unsqueeze(1)


def positive_logit_from_output(output: torch.Tensor) -> torch.Tensor:
    """
    Extract the class-1 logit/score from model output.
    """
    if isinstance(output, (tuple, list)):
        output = output[0]

    if output.ndim == 1:
        return output

    if output.ndim == 2 and output.shape[1] == 1:
        return output[:, 0]

    if output.ndim == 2 and output.shape[1] >= 2:
        return output[:, TARGET_LABEL]

    raise ValueError(f"Unsupported model output shape: {tuple(output.shape)}")


@torch.no_grad()
def predict_proba(model, X_raw: np.ndarray, norm_mean: np.ndarray, norm_std: np.ndarray, batch_size: int = 256) -> np.ndarray:
    """
    Predict class-1 probability using train-derived normalization.
    """
    model.eval()

    X_norm = normalize_trials(X_raw, norm_mean, norm_std)
    probs = []

    for start in range(0, len(X_norm), batch_size):
        xb = torch.as_tensor(X_norm[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        xb_in = prepare_model_input(xb)

        logits = model(xb_in)

        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        if logits.ndim == 2 and logits.shape[1] >= 2:
            prob = torch.softmax(logits, dim=1)[:, TARGET_LABEL]
        elif logits.ndim == 2 and logits.shape[1] == 1:
            prob = torch.sigmoid(logits[:, 0])
        elif logits.ndim == 1:
            prob = torch.sigmoid(logits)
        else:
            raise ValueError(f"Unsupported model output shape: {tuple(logits.shape)}")

        probs.append(prob.detach().cpu().numpy())

    return np.concatenate(probs, axis=0)

In [ ]:
def make_ig_baseline(x_norm: np.ndarray, times: np.ndarray) -> np.ndarray:
    """
    Build the Integrated Gradients baseline in normalized input space.
    """
    if BASELINE_MODE == "zeros":
        return np.zeros_like(x_norm, dtype=np.float32)

    if BASELINE_MODE == "prestim_mean":
        prestim_mask = get_window_mask(times, PRESTIM_WINDOW[0], PRESTIM_WINDOW[1])

        if prestim_mask.sum() == 0:
            print(
                "[WARN] PRESTIM_WINDOW has no samples after cropping. "
                "Falling back to zero baseline."
            )
            return np.zeros_like(x_norm, dtype=np.float32)

        mean_prestim = x_norm[:, :, prestim_mask].mean(axis=2, keepdims=True)
        return np.repeat(mean_prestim, x_norm.shape[2], axis=2).astype(np.float32)

    raise ValueError(f"Unsupported BASELINE_MODE: {BASELINE_MODE}")


def integrated_gradients_batch(
        model,
        x_batch_raw: np.ndarray,
        norm_mean: np.ndarray,
        norm_std: np.ndarray,
        times: np.ndarray,
        steps: int = 32,
) -> np.ndarray:
    """
    Compute Integrated Gradients for a batch.

    Input:
        x_batch_raw: (N, C, T), unnormalized EEG trials

    Output:
        attribution: (N, C, T), in normalized-input space
    """
    model.eval()

    x_norm_np = normalize_trials(x_batch_raw, norm_mean, norm_std)
    baseline_np = make_ig_baseline(x_norm_np, times)

    x = torch.as_tensor(x_norm_np, dtype=torch.float32, device=DEVICE)
    baseline = torch.as_tensor(baseline_np, dtype=torch.float32, device=DEVICE)

    total_grad = torch.zeros_like(x)

    alphas = torch.linspace(0.0, 1.0, int(steps), device=DEVICE)

    for alpha in alphas:
        xi = baseline + alpha * (x - baseline)
        xi.requires_grad_(True)

        model.zero_grad(set_to_none=True)

        out = model(prepare_model_input(xi))
        score = positive_logit_from_output(out).sum()

        grad = torch.autograd.grad(score, xi, retain_graph=False, create_graph=False)[0]
        total_grad += grad.detach()

    avg_grad = total_grad / float(steps)
    ig = (x - baseline) * avg_grad

    return ig.detach().cpu().numpy().astype(np.float32)


def get_safe_bands(bands: dict[str, tuple[float, float]], sfreq: float) -> dict[str, tuple[float, float]]:
    safe = {}
    nyq = float(sfreq) / 2.0

    for name, (lo, hi) in bands.items():
        lo = float(lo)
        hi = min(float(hi), nyq - 1.0)

        if hi <= lo:
            print(f"[WARN] Skipping band {name}: invalid after Nyquist adjustment.")
            continue

        safe[name] = (lo, hi)

    return safe


def bandpass_filter_trials(
        X: np.ndarray,
        sfreq: float,
        l_freq: float,
        h_freq: float,
        order: int = 4,
) -> np.ndarray:
    """
    Bandpass filter trials along the time axis.
    """
    nyq = float(sfreq) / 2.0

    lo = max(float(l_freq), 1e-3)
    hi = min(float(h_freq), nyq - 1.0)

    if hi <= lo:
        raise ValueError(f"Invalid band after Nyquist adjustment: {(lo, hi)}")

    sos = butter(
        int(order),
        [lo / nyq, hi / nyq],
        btype="bandpass",
        output="sos",
    )

    return sosfiltfilt(sos, X, axis=-1).astype(np.float32)

In [ ]:
def select_trials_for_subject(
        model,
        X: np.ndarray,
        y: np.ndarray,
        norm_mean: np.ndarray,
        norm_std: np.ndarray,
        min_correct_targets: int = 12,
        max_trials: int = 64,
        precomputed_prob: np.ndarray | None = None,
        precomputed_pred: np.ndarray | None = None,
):
    """
    Selection priority:
    1. correctly predicted target trials, if enough are available
    2. all target trials
    3. correctly predicted trials of either class
    4. all held-out trials
    """
    y = np.asarray(y).astype(int)

    if precomputed_prob is None or precomputed_pred is None:
        probs = predict_proba(model, X, norm_mean=norm_mean, norm_std=norm_std)
        preds = (probs >= PROB_THRESHOLD).astype(int)
    else:
        probs = np.asarray(precomputed_prob, dtype=float)
        preds = np.asarray(precomputed_pred).astype(int)

    idx_target = np.where(y == TARGET_LABEL)[0]
    idx_correct_target = idx_target[preds[idx_target] == TARGET_LABEL]

    if len(idx_correct_target) >= int(min_correct_targets):
        selected = idx_correct_target
        rule = "correct_target_only"
    elif len(idx_target) > 0:
        selected = idx_target
        rule = "all_target_trials"
    else:
        idx_correct_any = np.where(preds == y)[0]

        if len(idx_correct_any) > 0:
            selected = idx_correct_any
            rule = "correct_trials_any_class"
        else:
            selected = np.arange(len(y))
            rule = "all_trials"

    selected = pick_evenly_spaced_indices(selected, max_trials)

    summary = {
        "n_total_trials": int(len(X)),
        "n_target_trials": int(len(idx_target)),
        "n_correct_target_trials": int(len(idx_correct_target)),
        "n_selected_trials": int(len(selected)),
        "selection_rule": rule,
        "mean_prob_all": float(np.mean(probs)) if len(probs) else np.nan,
        "mean_prob_selected": float(np.mean(probs[selected])) if len(selected) else np.nan,
        "selected_positive_rate": float(np.mean(y[selected])) if len(selected) else np.nan,
    }

    return selected, probs, preds, summary

In [ ]:
def build_mne_info(ch_names: list[str], sfreq: float) -> mne.Info:
    info = mne.create_info(
        ch_names=list(ch_names),
        sfreq=float(sfreq),
        ch_types="eeg",
    )

    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, on_missing="ignore")

    return info


def plot_topomap_safe(values, ch_names, sfreq, ax, title: str):
    """
    Plot a topomap when valid channel positions exist.
    Otherwise place a readable placeholder.
    """
    values = np.asarray(values, dtype=float)

    try:
        info = build_mne_info(ch_names, sfreq)
        picks = valid_topomap_picks(info)

        if len(picks) == 0:
            raise RuntimeError("No valid topomap channel positions.")

        info_valid = mne.pick_info(info, picks, copy=True)
        values_valid = values[picks]

        mne.viz.plot_topomap(
            values_valid,
            info_valid,
            axes=ax,
            show=False,
            contours=6,
        )

        ax.set_title(title)

    except Exception as e:
        ax.text(
            0.5,
            0.5,
            f"Topomap unavailable\n{type(e).__name__}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title(title)
        ax.axis("off")


def top_channels_from_map(
        map_ct: np.ndarray,
        ch_names: list[str],
        times: np.ndarray,
        top_k: int = 12,
) -> pd.DataFrame:
    """
    Rank channels by mean absolute attribution in the P300 window.
    """
    map_ct = np.asarray(map_ct, dtype=float)
    p300_mask = p300_mask_from_times(times)

    importance_full = map_ct.mean(axis=1)
    importance_p300 = map_ct[:, p300_mask].mean(axis=1)

    df = pd.DataFrame({
        "channel": list(ch_names),
        "importance_full": importance_full,
        "importance_p300": importance_p300,
    })

    return (
        df.sort_values("importance_p300", ascending=False)
        .head(int(top_k))
        .reset_index(drop=True)
    )


def time_peaks_from_map(
        map_ct: np.ndarray,
        times: np.ndarray,
        top_k: int = 10,
) -> pd.DataFrame:
    """
    Rank time points by mean absolute attribution across channels.
    """
    map_ct = np.asarray(map_ct, dtype=float)
    time_curve = map_ct.mean(axis=0)

    peak_idx = np.argsort(time_curve)[::-1][:int(top_k)]

    return (
        pd.DataFrame({
            "time_s": np.asarray(times)[peak_idx],
            "importance": time_curve[peak_idx],
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
def save_broadband_summary_figure(
        map_ct: np.ndarray,
        ch_names: list[str],
        times: np.ndarray,
        sfreq: float,
        out_path: Path,
        title_prefix: str,
):
    p300_mask = p300_mask_from_times(times)

    time_curve = map_ct.mean(axis=0)
    p300_topo = map_ct[:, p300_mask].mean(axis=1)
    top_df = top_channels_from_map(map_ct, ch_names, times, top_k=12)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Time importance
    axes[0].plot(times, time_curve, linewidth=2)
    axes[0].axvline(BASELINE_TIME, linestyle="--", linewidth=1)
    axes[0].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
    axes[0].set_title("Time attribution")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Mean |IG attribution|")
    axes[0].grid(True, alpha=0.3)

    # Top channels
    plot_df = top_df.iloc[::-1]
    axes[1].barh(plot_df["channel"], plot_df["importance_p300"])
    axes[1].set_title("Top channels in P300 window")
    axes[1].set_xlabel("Mean |IG attribution|")

    # Topomap
    plot_topomap_safe(
        p300_topo,
        ch_names=ch_names,
        sfreq=sfreq,
        ax=axes[2],
        title="P300-window topography",
    )

    fig.suptitle(title_prefix, fontsize=15)
    fig.tight_layout()

    save_figure(fig, out_path)
    plt.close(fig)


def save_trial_consistency_figure(
        attr_trials: np.ndarray,
        times: np.ndarray,
        out_path: Path,
        title_prefix: str,
):
    trial_time = np.abs(attr_trials).mean(axis=1)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 5),
        gridspec_kw={"width_ratios": [1.2, 1.0]},
    )

    im = axes[0].imshow(
        trial_time,
        aspect="auto",
        origin="lower",
        extent=[times[0], times[-1], 0, len(trial_time)],
    )
    axes[0].axvline(BASELINE_TIME, linestyle="--", linewidth=1)
    axes[0].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
    axes[0].set_title("Trial-by-time attribution")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Selected trial index")
    cbar = fig.colorbar(im, ax=axes[0])
    cbar.set_label("Mean |IG attribution|")

    axes[1].plot(times, trial_time.mean(axis=0), linewidth=2)
    axes[1].axvline(BASELINE_TIME, linestyle="--", linewidth=1)
    axes[1].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
    axes[1].set_title("Mean across selected trials")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Mean |IG attribution|")
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title_prefix, fontsize=15)
    fig.tight_layout()

    save_figure(fig, out_path)
    plt.close(fig)


def save_band_limited_figure(
        band_maps: dict[str, np.ndarray],
        ch_names: list[str],
        times: np.ndarray,
        sfreq: float,
        out_path: Path,
        title_prefix: str,
):
    band_names = list(band_maps.keys())

    if len(band_names) == 0:
        raise ValueError("No band maps available to plot.")

    n_bands = len(band_names)

    fig, axes = plt.subplots(
        2,
        n_bands,
        figsize=(5 * n_bands, 8),
        squeeze=False,
    )

    p300_mask = p300_mask_from_times(times)

    for col, band_name in enumerate(band_names):
        band_map = band_maps[band_name]

        band_time = band_map.mean(axis=0)
        band_topo = band_map[:, p300_mask].mean(axis=1)

        ax_time = axes[0, col]
        ax_time.plot(times, band_time, linewidth=2)
        ax_time.axvline(BASELINE_TIME, linestyle="--", linewidth=1)
        ax_time.axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
        ax_time.set_title(f"{band_name.capitalize()} time attribution")
        ax_time.set_xlabel("Time (s)")
        ax_time.set_ylabel("Mean |IG attribution|")
        ax_time.grid(True, alpha=0.3)

        ax_topo = axes[1, col]
        plot_topomap_safe(
            band_topo,
            ch_names=ch_names,
            sfreq=sfreq,
            ax=ax_topo,
            title=f"{band_name.capitalize()} P300 topography",
        )

    fig.suptitle(title_prefix, fontsize=15)
    fig.tight_layout()

    save_figure(fig, out_path)
    plt.close(fig)

In [ ]:
selection_rows = []
subject_metric_rows = []
band_summary_rows = []
broadband_top_channel_rows = []
broadband_peak_rows = []

subject_broadband_maps = {}
subject_band_maps = {}
subject_time_profiles = {}

reference_ch_names = None
reference_times = None
reference_sfreq = None

for sid, subj_cfg in sorted(SUBJECT_CONFIGS.items()):
    print("\n" + "=" * 90)
    print(f"LOSO held-out subject: sub-{sid}")
    print("=" * 90)

    cache_path = Path(subj_cfg["heldout_eval_path"])
    ckpt_path = Path(subj_cfg["checkpoint_path"])
    chosen_seed = int(subj_cfg["seed"])

    if not cache_path.exists():
        print(f"[SKIP] Missing held-out cache: {cache_path}")
        continue

    if not ckpt_path.exists():
        print(f"[SKIP] Missing checkpoint: {ckpt_path}")
        continue

    eval_data = load_eval_data_from_npz(cache_path)

    X = eval_data["X"]
    y = eval_data["y"]
    subjects = eval_data["subjects"]
    times = eval_data["times"]
    ch_names = eval_data["ch_names"]
    sfreq = eval_data["sfreq"]

    prob = eval_data["prob"]
    pred = eval_data["pred"]

    keep = subjects == sid

    if keep.any():
        X_sub = X[keep]
        y_sub = y[keep]
        prob_sub = prob[keep] if prob is not None else None
        pred_sub = pred[keep] if pred is not None else None
    else:
        X_sub = X
        y_sub = y
        prob_sub = prob
        pred_sub = pred

    print("Chosen seed      :", chosen_seed)
    print("Selected seed AUC:", subj_cfg["auc"])
    print("X_sub shape      :", X_sub.shape)
    print("Target count     :", int((y_sub == TARGET_LABEL).sum()))

    if reference_ch_names is None:
        reference_ch_names = list(ch_names)
        reference_times = np.asarray(times, dtype=float)
        reference_sfreq = float(sfreq)
    else:
        if list(ch_names) != reference_ch_names:
            raise ValueError(f"Channel mismatch for subject {sid}")
        if len(times) != len(reference_times) or not np.allclose(times, reference_times):
            raise ValueError(f"Time-axis mismatch for subject {sid}")

    model, ckpt, norm_mean, norm_std = load_model_and_norm(
        ckpt_path,
        n_channels=X_sub.shape[1],
        n_times=X_sub.shape[2],
    )

    selected_idx, probs, preds, sel_summary = select_trials_for_subject(
        model=model,
        X=X_sub,
        y=y_sub,
        norm_mean=norm_mean,
        norm_std=norm_std,
        min_correct_targets=MIN_CORRECT_TARGETS,
        max_trials=MAX_TRIALS_PER_SUBJECT,
        precomputed_prob=prob_sub,
        precomputed_pred=pred_sub,
    )

    if len(selected_idx) == 0:
        print(f"[SKIP] No selected trials for sub-{sid}")
        continue

    X_sel = X_sub[selected_idx]
    y_sel = y_sub[selected_idx]

    print("Selection rule   :", sel_summary["selection_rule"])
    print("Selected trials  :", len(X_sel))

    subj_fig_root = FIG_ROOT / f"sub-{sid}"
    subj_fig_root.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------
    # Broadband Integrated Gradients
    # ------------------------------------------------------------
    attr_batches = []

    for start in range(0, len(X_sel), ATTR_BATCH_SIZE):
        xb = X_sel[start:start + ATTR_BATCH_SIZE]

        attr = integrated_gradients_batch(
            model=model,
            x_batch_raw=xb,
            norm_mean=norm_mean,
            norm_std=norm_std,
            times=times,
            steps=IG_STEPS,
        )

        attr_batches.append(attr)

    attr_trials = np.concatenate(attr_batches, axis=0)
    broadband_map = np.abs(attr_trials).mean(axis=0)

    subject_broadband_maps[sid] = broadband_map
    subject_time_profiles[sid] = broadband_map.mean(axis=0)

    save_broadband_summary_figure(
        map_ct=broadband_map,
        ch_names=ch_names,
        times=times,
        sfreq=sfreq,
        out_path=subj_fig_root / f"sub-{sid}_eegnet_loso_broadband_interpretability.png",
        title_prefix=f"EEGNet LOSO broadband interpretability — sub-{sid}",
    )

    save_trial_consistency_figure(
        attr_trials=attr_trials,
        times=times,
        out_path=subj_fig_root / f"sub-{sid}_eegnet_loso_trial_consistency.png",
        title_prefix=f"EEGNet LOSO trial consistency — sub-{sid}",
    )

    # ------------------------------------------------------------
    # Band-limited diagnostics
    # ------------------------------------------------------------
    safe_bands = get_safe_bands(BANDS, sfreq)
    band_maps = {}

    for band_name, (lo, hi) in safe_bands.items():
        print(f"Band attribution: {band_name} ({lo}-{hi} Hz)")

        X_band = bandpass_filter_trials(
            X_sel,
            sfreq=sfreq,
            l_freq=lo,
            h_freq=hi,
        )

        band_attr_batches = []

        for start in range(0, len(X_band), ATTR_BATCH_SIZE):
            xb = X_band[start:start + ATTR_BATCH_SIZE]

            attr = integrated_gradients_batch(
                model=model,
                x_batch_raw=xb,
                norm_mean=norm_mean,
                norm_std=norm_std,
                times=times,
                steps=IG_STEPS,
            )

            band_attr_batches.append(attr)

        band_attr = np.concatenate(band_attr_batches, axis=0)
        band_map = np.abs(band_attr).mean(axis=0)

        band_maps[band_name] = band_map

        band_summary_rows.append({
            "subject": sid,
            "seed": chosen_seed,
            "selected_seed_auc": float(subj_cfg["auc"]),
            "band": band_name,
            "l_freq": float(lo),
            "h_freq": float(hi),
            "importance_mean": float(band_map.mean()),
            "importance_std": float(band_map.std()),
        })

        top_band_df = top_channels_from_map(
            band_map,
            ch_names=ch_names,
            times=times,
            top_k=12,
        )
        top_band_df.insert(0, "subject", sid)
        top_band_df.insert(1, "seed", chosen_seed)
        top_band_df.insert(2, "band", band_name)

        save_dataframe(
            top_band_df,
            TABLE_ROOT / f"sub-{sid}_{band_name}_loso_band_top_channels.csv",
        )

    subject_band_maps[sid] = band_maps

    save_band_limited_figure(
        band_maps=band_maps,
        ch_names=ch_names,
        times=times,
        sfreq=sfreq,
        out_path=subj_fig_root / f"sub-{sid}_eegnet_loso_band_limited_diagnostic.png",
        title_prefix=f"EEGNet LOSO band-limited diagnostic — sub-{sid}",
    )

    # ------------------------------------------------------------
    # Subject-level tables
    # ------------------------------------------------------------
    top_df = top_channels_from_map(
        broadband_map,
        ch_names=ch_names,
        times=times,
        top_k=12,
    )
    top_df.insert(0, "subject", sid)
    top_df.insert(1, "seed", chosen_seed)
    top_df.insert(2, "selected_seed_auc", float(subj_cfg["auc"]))

    peak_df = time_peaks_from_map(
        broadband_map,
        times=times,
        top_k=10,
    )
    peak_df.insert(0, "subject", sid)
    peak_df.insert(1, "seed", chosen_seed)
    peak_df.insert(2, "selected_seed_auc", float(subj_cfg["auc"]))

    time_curve_df = pd.DataFrame({
        "subject": sid,
        "seed": chosen_seed,
        "selected_seed_auc": float(subj_cfg["auc"]),
        "time_s": times,
        "mean_abs_attribution": broadband_map.mean(axis=0),
    })

    save_dataframe(top_df, TABLE_ROOT / f"sub-{sid}_loso_broadband_top_channels.csv")
    save_dataframe(peak_df, TABLE_ROOT / f"sub-{sid}_loso_broadband_time_peaks.csv")
    save_dataframe(time_curve_df, TABLE_ROOT / f"sub-{sid}_loso_broadband_time_importance.csv")

    broadband_top_channel_rows.append(top_df.copy())
    broadband_peak_rows.append(peak_df.copy())

    p300_mask = p300_mask_from_times(times)

    selection_rows.append({
        "subject": sid,
        "seed": chosen_seed,
        "selected_seed_auc": float(subj_cfg["auc"]),
        "cache_path": str(cache_path),
        "checkpoint_path": str(ckpt_path),
        **sel_summary,
    })

    subject_metric_rows.append({
        "subject": sid,
        "seed": chosen_seed,
        "selected_seed_auc": float(subj_cfg["auc"]),
        "n_selected_trials": int(len(X_sel)),
        "selection_rule": sel_summary["selection_rule"],
        "broadband_mean_attr": float(broadband_map.mean()),
        "broadband_std_attr": float(broadband_map.std()),
        "p300_mean_attr": float(broadband_map[:, p300_mask].mean()),
        "p300_max_attr": float(broadband_map[:, p300_mask].max()),
    })

    print(f"Saved outputs for sub-{sid}")

In [ ]:
def safe_dataframe(rows, required_cols=None, sort_cols=None) -> pd.DataFrame:
    df = pd.DataFrame(rows)

    if required_cols is not None:
        for col in required_cols:
            if col not in df.columns:
                df[col] = pd.Series(dtype="object")

    if sort_cols is not None and len(df) > 0:
        valid = [c for c in sort_cols if c in df.columns]

        if valid:
            df = df.sort_values(valid).reset_index(drop=True)

    return df


selection_df = safe_dataframe(
    selection_rows,
    required_cols=[
        "subject",
        "seed",
        "selected_seed_auc",
        "cache_path",
        "checkpoint_path",
        "n_total_trials",
        "n_target_trials",
        "n_correct_target_trials",
        "n_selected_trials",
        "selection_rule",
        "mean_prob_all",
        "mean_prob_selected",
        "selected_positive_rate",
    ],
    sort_cols=["subject"],
)

subject_metric_df = safe_dataframe(
    subject_metric_rows,
    required_cols=[
        "subject",
        "seed",
        "selected_seed_auc",
        "n_selected_trials",
        "selection_rule",
        "broadband_mean_attr",
        "broadband_std_attr",
        "p300_mean_attr",
        "p300_max_attr",
    ],
    sort_cols=["subject"],
)

band_summary_df = safe_dataframe(
    band_summary_rows,
    required_cols=[
        "subject",
        "seed",
        "selected_seed_auc",
        "band",
        "l_freq",
        "h_freq",
        "importance_mean",
        "importance_std",
    ],
    sort_cols=["subject", "band"],
)

if broadband_top_channel_rows:
    broadband_top_channels_df = pd.concat(
        broadband_top_channel_rows,
        axis=0,
        ignore_index=True,
    )
else:
    broadband_top_channels_df = pd.DataFrame(
        columns=[
            "subject",
            "seed",
            "selected_seed_auc",
            "channel",
            "importance_full",
            "importance_p300",
        ]
    )

if broadband_peak_rows:
    broadband_peaks_df = pd.concat(
        broadband_peak_rows,
        axis=0,
        ignore_index=True,
    )
else:
    broadband_peaks_df = pd.DataFrame(
        columns=[
            "subject",
            "seed",
            "selected_seed_auc",
            "time_s",
            "importance",
        ]
    )

save_dataframe(selection_df, TABLE_ROOT / "eegnet_loso_trial_selection_summary.csv")
save_dataframe(subject_metric_df, TABLE_ROOT / "eegnet_loso_subject_metric_summary.csv")
save_dataframe(band_summary_df, TABLE_ROOT / "eegnet_loso_subject_band_importance_summary.csv")
save_dataframe(broadband_top_channels_df, TABLE_ROOT / "eegnet_loso_subject_broadband_top_channels.csv")
save_dataframe(broadband_peaks_df, TABLE_ROOT / "eegnet_loso_subject_broadband_time_peaks.csv")

print("Saved subject-level summary tables.")
print("selection_df shape             :", selection_df.shape)
print("subject_metric_df shape        :", subject_metric_df.shape)
print("band_summary_df shape          :", band_summary_df.shape)
print("broadband_top_channels_df shape:", broadband_top_channels_df.shape)
print("broadband_peaks_df shape       :", broadband_peaks_df.shape)

display(round_numeric_columns(selection_df, 4))
display(round_numeric_columns(subject_metric_df, 4))
display(round_numeric_columns(band_summary_df, 4))

In [ ]:
if not subject_broadband_maps:
    raise RuntimeError(
        "No subject maps were created. Check manifest paths, checkpoints, and held-out caches."
    )

subject_ids_sorted = sorted(subject_broadband_maps.keys())

ch_names_summary = reference_ch_names
times_summary = reference_times
sfreq_summary = reference_sfreq

grand_broadband_stack = np.stack(
    [subject_broadband_maps[sid] for sid in subject_ids_sorted],
    axis=0,
)
grand_broadband_map = grand_broadband_stack.mean(axis=0)

grand_band_maps = {}

available_band_names = sorted(
    set.intersection(
        *[set(subject_band_maps[sid].keys()) for sid in subject_ids_sorted]
    )
)

for band_name in available_band_names:
    band_stack = np.stack(
        [subject_band_maps[sid][band_name] for sid in subject_ids_sorted],
        axis=0,
    )
    grand_band_maps[band_name] = band_stack.mean(axis=0)

subject_time_matrix = np.stack(
    [subject_time_profiles[sid] for sid in subject_ids_sorted],
    axis=0,
)

save_broadband_summary_figure(
    map_ct=grand_broadband_map,
    ch_names=ch_names_summary,
    times=times_summary,
    sfreq=sfreq_summary,
    out_path=SUMMARY_FIG_ROOT / "chapter7_eegnet_loso_broadband_interpretability.png",
    title_prefix="EEGNet LOSO broadband interpretability summary",
)

save_band_limited_figure(
    band_maps=grand_band_maps,
    ch_names=ch_names_summary,
    times=times_summary,
    sfreq=sfreq_summary,
    out_path=SUMMARY_FIG_ROOT / "chapter7_eegnet_loso_band_limited_diagnostic.png",
    title_prefix="EEGNet LOSO band-limited attribution diagnostic",
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(20, 7),
    gridspec_kw={"width_ratios": [1.05, 1.15]},
)

im = axes[0].imshow(
    subject_time_matrix,
    aspect="auto",
    origin="lower",
    extent=[times_summary[0], times_summary[-1], 0, len(subject_ids_sorted)],
)

axes[0].axvline(BASELINE_TIME, linestyle="--", linewidth=1)
axes[0].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
axes[0].set_title("Subject-by-time LOSO attribution heatmap")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Held-out subject")
axes[0].set_yticks(np.arange(len(subject_ids_sorted)) + 0.5)
axes[0].set_yticklabels([f"sub-{sid}" for sid in subject_ids_sorted])

cbar = fig.colorbar(im, ax=axes[0])
cbar.set_label("Mean |IG attribution|")

axes[1].plot(
    times_summary,
    subject_time_matrix.mean(axis=0),
    linewidth=2.5,
)
axes[1].axvline(BASELINE_TIME, linestyle="--", linewidth=1)
axes[1].axvspan(P300_WINDOW[0], P300_WINDOW[1], alpha=0.15)
axes[1].set_title("Mean LOSO attribution across held-out subjects")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Mean |IG attribution|")
axes[1].grid(True, alpha=0.3)

fig.suptitle("EEGNet LOSO attribution consistency summary", fontsize=16)
fig.tight_layout()

save_figure(
    fig,
    SUMMARY_FIG_ROOT / "chapter7_eegnet_loso_trial_consistency_summary.png",
)

plt.close(fig)

print("Saved grand LOSO summary figures to:")
print(SUMMARY_FIG_ROOT)

In [ ]:
grand_top_df = top_channels_from_map(
    grand_broadband_map,
    ch_names=ch_names_summary,
    times=times_summary,
    top_k=12,
)
save_dataframe(
    grand_top_df,
    TABLE_ROOT / "eegnet_loso_grand_broadband_top_channels.csv",
)

grand_peak_df = time_peaks_from_map(
    grand_broadband_map,
    times=times_summary,
    top_k=10,
)
save_dataframe(
    grand_peak_df,
    TABLE_ROOT / "eegnet_loso_grand_broadband_time_peaks.csv",
)

grand_time_curve_df = pd.DataFrame({
    "time_s": times_summary,
    "mean_abs_attribution": grand_broadband_map.mean(axis=0),
})
save_dataframe(
    grand_time_curve_df,
    TABLE_ROOT / "eegnet_loso_grand_broadband_time_importance.csv",
)

grand_band_rows = []

for band_name, band_map in grand_band_maps.items():
    grand_band_rows.append({
        "band": band_name,
        "importance_mean": float(band_map.mean()),
        "importance_std": float(band_map.std()),
    })

    band_top_df = top_channels_from_map(
        band_map,
        ch_names=ch_names_summary,
        times=times_summary,
        top_k=12,
    )
    band_top_df.insert(0, "band", band_name)

    save_dataframe(
        band_top_df,
        TABLE_ROOT / f"eegnet_loso_grand_{band_name}_top_channels.csv",
    )

grand_band_df = (
    pd.DataFrame(grand_band_rows)
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

save_dataframe(
    grand_band_df,
    TABLE_ROOT / "eegnet_loso_grand_band_importance_summary.csv",
)

summary_manifest = {
    "experiment_key": EXPERIMENT_NAME,
    "export_manifest_path": str(MANIFEST_PATH),
    "interpretability_output_dir": str(OUT_DIR),
    "n_subjects": int(len(subject_ids_sorted)),
    "subjects": subject_ids_sorted,
    "seed_selection": SEED_SELECTION,
    "baseline_mode": BASELINE_MODE,
    "ig_steps": int(IG_STEPS),
    "max_trials_per_subject": int(MAX_TRIALS_PER_SUBJECT),
    "p300_window": list(P300_WINDOW),
    "prestim_window": list(PRESTIM_WINDOW),
    "bands": BANDS,
}

save_json(
    summary_manifest,
    TABLE_ROOT / "eegnet_loso_interpretability_manifest.json",
)

print("Saved grand summary tables.")
display(round_numeric_columns(grand_top_df, 4))
display(round_numeric_columns(grand_peak_df, 4))
display(round_numeric_columns(grand_band_df, 4))

In [ ]:
print("Per-subject figure folders:")
for p in sorted(FIG_ROOT.glob("sub-*")):
    if p.is_dir():
        print(" -", p)

print("\nSummary figures:")
for p in sorted(SUMMARY_FIG_ROOT.glob("*.png")):
    print(" -", p)

print("\nKey summary tables:")
key_tables = [
    "eegnet_loso_trial_selection_summary.csv",
    "eegnet_loso_subject_metric_summary.csv",
    "eegnet_loso_subject_band_importance_summary.csv",
    "eegnet_loso_subject_broadband_top_channels.csv",
    "eegnet_loso_subject_broadband_time_peaks.csv",
    "eegnet_loso_grand_broadband_top_channels.csv",
    "eegnet_loso_grand_broadband_time_peaks.csv",
    "eegnet_loso_grand_broadband_time_importance.csv",
    "eegnet_loso_grand_band_importance_summary.csv",
    "eegnet_loso_interpretability_manifest.json",
]

for name in key_tables:
    p = TABLE_ROOT / name
    print(" -", p.name, "| exists =", p.exists())

print("\nNotebook 13 complete.")
print("Output directory:", OUT_DIR)